# Zotero API: Highlights aus PDF extrahieren

Dieses Notebook ermöglicht dir:
- Alle PDF-Attachments aus deiner Zotero-Bibliothek abzurufen
- Eine **Item-ID** einzugeben
- **Alle Highlights** mit **vollständigen Metadaten** (Text, Seite, Tags, Kommentar, Farbe, etc.) anzuzeigen

> **Voraussetzung:** Du brauchst einen [Zotero API-Schlüssel](https://www.zotero.org/settings/keys) mit Lesezugriff.

In [1]:
!pip install requests pandas ipywidgets -q


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [8]:
import requests
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import json
from datetime import datetime

# --- Konfiguration ---
API_BASE = "https://api.zotero.org"

# Eingabefelder für API-Zugang
user_id_input = widgets.Text(
    value='', 
    placeholder='Deine Zotero User ID (z. B. 1234567)',
    description='User ID:',
    layout=widgets.Layout(width='50%')
)

api_key_input = widgets.Password(
    value='', 
    placeholder='Dein API-Schlüssel',
    description='API-Key:',
    layout=widgets.Layout(width='50%')
)

load_structure_button = widgets.Button(description="Struktur laden", button_style='primary')
output_config = widgets.Output()

display(user_id_input, api_key_input, load_structure_button, output_config)

# Globale Variablen
all_items = []
headers = {}
user_id = ""
collections = []
groups = []
current_source = ""

# Widgets für Sammlungen/Gruppen
collection_dropdown = widgets.Dropdown(
    options=[],
    description='Sammlung:',
    layout=widgets.Layout(width='70%'),
    disabled=True
)

group_dropdown = widgets.Dropdown(
    options=[],
    description='Gruppe:',
    layout=widgets.Layout(width='70%'),
    disabled=True
)

load_items_button = widgets.Button(description="Items laden", button_style='success', disabled=True)
output_items = widgets.Output()

# Widget für Item-Auswahl
item_dropdown = widgets.Dropdown(
    options=[],
    description='Item wählen:',
    layout=widgets.Layout(width='90%'),
    disabled=True
)

load_annotations_button = widgets.Button(description="Annotationen laden", button_style='info', disabled=True)
output_annotations = widgets.Output()

def load_structure(b):
    """Lädt Gruppen und Sammlungen"""
    with output_config:
        clear_output()
        global user_id, headers, collections, groups
        
        user_id = user_id_input.value.strip()
        api_key = api_key_input.value.strip()
        
        if not user_id or not api_key:
            print("Bitte User ID und API-Key eingeben!")
            return
        
        headers = {"Zotero-API-Key": api_key}
        
        # Lade Gruppen
        print("Lade Gruppen...")
        try:
            response = requests.get(f"{API_BASE}/users/{user_id}/groups", headers=headers)
            if response.status_code == 200:
                groups_data = response.json()
                groups = [("Meine Bibliothek (User)", f"user_{user_id}")]
                for group in groups_data:
                    groups.append((group["data"]["name"], f"group_{group['id']}"))
                print(f"✓ {len(groups)-1} Gruppen gefunden")
            else:
                print(f"Fehler beim Laden der Gruppen: {response.status_code}")
                groups = [("Meine Bibliothek (User)", f"user_{user_id}")]
        except Exception as e:
            print(f"Fehler: {e}")
            groups = [("Meine Bibliothek (User)", f"user_{user_id}")]
        
        # Lade Sammlungen der User-Bibliothek
        print("Lade Sammlungen...")
        try:
            response = requests.get(f"{API_BASE}/users/{user_id}/collections", headers=headers)
            if response.status_code == 200:
                collections_data = response.json()
                collections = [("Alle Items (keine Sammlung)", "")]
                for coll in collections_data:
                    collections.append((coll["data"]["name"], coll["key"]))
                print(f"✓ {len(collections)-1} Sammlungen gefunden")
            else:
                print(f"Fehler beim Laden der Sammlungen: {response.status_code}")
                collections = [("Alle Items (keine Sammlung)", "")]
        except Exception as e:
            print(f"Fehler: {e}")
            collections = [("Alle Items (keine Sammlung)", "")]
        
        # Aktualisiere Dropdowns
        group_dropdown.options = groups
        group_dropdown.disabled = False
        collection_dropdown.options = collections
        collection_dropdown.disabled = False
        load_items_button.disabled = False
        
        print("\n✓ Fertig! Wähle jetzt Gruppe und/oder Sammlung aus.")
        
        # Zeige Auswahl-Widgets
        display(HTML("<hr><h4>📁 Wähle Quelle:</h4>"))
        display(group_dropdown)
        display(collection_dropdown)
        display(load_items_button)
        display(output_items)

def update_collections(change):
    """Lädt Sammlungen wenn Gruppe geändert wird"""
    global collections
    selected = group_dropdown.value
    
    if not selected:
        return
    
    with output_config:
        print("\nLade Sammlungen für gewählte Quelle...")
    
    try:
        if selected.startswith("user_"):
            uid = selected.replace("user_", "")
            response = requests.get(f"{API_BASE}/users/{uid}/collections", headers=headers)
        else:  # group_
            gid = selected.replace("group_", "")
            response = requests.get(f"{API_BASE}/groups/{gid}/collections", headers=headers)
        
        if response.status_code == 200:
            collections_data = response.json()
            collections = [("Alle Items (keine Sammlung)", "")]
            for coll in collections_data:
                collections.append((coll["data"]["name"], coll["key"]))
            collection_dropdown.options = collections
            with output_config:
                print(f"✓ {len(collections)-1} Sammlungen geladen")
        else:
            collections = [("Alle Items (keine Sammlung)", "")]
            collection_dropdown.options = collections
    except Exception as e:
        with output_config:
            print(f"Fehler beim Laden der Sammlungen: {e}")

def load_items(b):
    """Lädt Items aus der gewählten Quelle/Sammlung"""
    with output_items:
        clear_output()
        global all_items, current_source
        
        selected_source = group_dropdown.value
        selected_collection = collection_dropdown.value
        
        if not selected_source:
            print("Bitte wähle eine Quelle aus!")
            return
        
        current_source = selected_source
        
        # Erstelle URL basierend auf Auswahl
        if selected_source.startswith("user_"):
            uid = selected_source.replace("user_", "")
            if selected_collection:
                url = f"{API_BASE}/users/{uid}/collections/{selected_collection}/items"
                print(f"Lade Items aus Sammlung '{collection_dropdown.label}'...")
            else:
                url = f"{API_BASE}/users/{uid}/items"
                print("Lade alle Items aus User-Bibliothek...")
        else:  # group_
            gid = selected_source.replace("group_", "")
            if selected_collection:
                url = f"{API_BASE}/groups/{gid}/collections/{selected_collection}/items"
                print(f"Lade Items aus Sammlung '{collection_dropdown.label}'...")
            else:
                url = f"{API_BASE}/groups/{gid}/items"
                print(f"Lade alle Items aus Gruppe '{group_dropdown.label}'...")
        
        params = {
            "limit": 100,
            "start": 0
        }
        
        all_items = []
        while True:
            response = requests.get(url, headers=headers, params=params)
            if response.status_code != 200:
                print(f"Fehler: {response.status_code} – {response.text}")
                return
            
            data = response.json()
            if not data:
                break
                
            for item in data:
                all_items.append({
                    "key": item["key"],
                    "title": item["data"].get("title", "Unbenannt"),
                    "itemType": item["data"].get("itemType", "unknown")
                })
            
            # Paginierung
            if len(data) < params["limit"]:
                break
            params["start"] += params["limit"]
        
        print(f"\n✓ {len(all_items)} Items gefunden.")
        
        # Erstelle Dropdown-Optionen für Items
        item_options = [(f"[{item['itemType']}] {item['title'][:80]}", item['key']) 
                       for item in all_items]
        item_dropdown.options = item_options
        item_dropdown.disabled = False
        load_annotations_button.disabled = False
        
        # Zeige Item-Auswahl
        display(HTML("<hr><h4>📄 Wähle ein Item für Annotationen:</h4>"))
        display(item_dropdown)
        display(load_annotations_button)
        display(output_annotations)

def load_annotations(b):
    """Lädt Annotationen für das ausgewählte Item"""
    with output_annotations:
        clear_output()
        
        selected_item_key = item_dropdown.value
        if not selected_item_key:
            print("Bitte wähle ein Item aus!")
            return
        
        print(f"Lade Annotationen für Item '{item_dropdown.label}'...")
        
        # Baue URL basierend auf der aktuellen Quelle
        if current_source.startswith("user_"):
            uid = current_source.replace("user_", "")
            base_url = f"{API_BASE}/users/{uid}"
        else:  # group_
            gid = current_source.replace("group_", "")
            base_url = f"{API_BASE}/groups/{gid}"
        
        # Methode 1: Finde alle Attachments (PDFs) für dieses Item
        print("  → Suche PDF-Attachments...")
        try:
            # Lade alle Items und filtere nach parentItem
            response = requests.get(f"{base_url}/items", headers=headers, params={
                "itemType": "attachment",
                "limit": 100
            })
            
            if response.status_code != 200:
                print(f"Fehler beim Laden der Attachments: {response.status_code}")
                return
            
            attachments = response.json()
            pdf_attachments = [att for att in attachments 
                              if att["data"].get("parentItem") == selected_item_key 
                              and att["data"].get("contentType") == "application/pdf"]
            
            print(f"  ✓ {len(pdf_attachments)} PDF(s) gefunden")
            
            if not pdf_attachments:
                print("\n⚠ Keine PDF-Attachments für dieses Item gefunden.")
                return
            
            # Methode 2: Lade alle Annotationen und filtere nach parentItem
            print("  → Lade Annotationen...")
            all_annotations = []
            
            for pdf_att in pdf_attachments:
                pdf_key = pdf_att["key"]
                pdf_title = pdf_att["data"].get("title", "Unbenannt")
                
                # Lade Annotationen für dieses PDF
                response = requests.get(f"{base_url}/items", headers=headers, params={
                    "itemType": "annotation",
                    "limit": 100
                })
                
                if response.status_code == 200:
                    annotations = response.json()
                    pdf_annotations = [ann for ann in annotations 
                                      if ann["data"].get("parentItem") == pdf_key]
                    
                    for ann in pdf_annotations:
                        ann_data = ann["data"]
                        all_annotations.append({
                            "pdf": pdf_title,
                            "type": ann_data.get("annotationType", "unknown"),
                            "text": ann_data.get("annotationText", ""),
                            "comment": ann_data.get("annotationComment", ""),
                            "color": ann_data.get("annotationColor", ""),
                            "pageLabel": ann_data.get("annotationPageLabel", ""),
                            "tags": [tag.get("tag", "") for tag in ann_data.get("tags", [])]
                        })
            
            print(f"  ✓ {len(all_annotations)} Annotationen gefunden\n")
            
            if not all_annotations:
                print("⚠ Keine Annotationen für dieses Item gefunden.")
                return
            
            # Zeige Annotationen übersichtlich
            for i, ann in enumerate(all_annotations, 1):
                html_output = f"""
                <div style="border: 2px solid #{'#' + ann['color'] if ann['color'] else 'cccccc'}; 
                            padding: 15px; margin: 10px 0; border-radius: 8px; 
                            background-color: #f9f9f9;">
                    <h4 style="margin-top: 0; color: #2c3e50;">
                        📌 Annotation {i} 
                        <span style="font-size: 0.8em; color: #7f8c8d;">
                            [{ann['type']}] - Seite {ann['pageLabel']}
                        </span>
                    </h4>
                    <p><strong>PDF:</strong> {ann['pdf']}</p>
                """
                
                if ann['text']:
                    html_output += f"""
                    <div style="background-color: #fff3cd; padding: 10px; border-left: 4px solid #ffc107; margin: 10px 0;">
                        <strong>📝 Markierter Text:</strong><br>
                        {ann['text']}
                    </div>
                    """
                
                if ann['comment']:
                    html_output += f"""
                    <div style="background-color: #d1ecf1; padding: 10px; border-left: 4px solid #17a2b8; margin: 10px 0;">
                        <strong>💬 Kommentar:</strong><br>
                        {ann['comment']}
                    </div>
                    """
                
                if ann['tags']:
                    tags_html = ", ".join([f"<span style='background-color: #e7f3ff; padding: 2px 8px; border-radius: 3px; margin: 2px;'>{tag}</span>" 
                                          for tag in ann['tags']])
                    html_output += f"<p><strong>🏷 Tags:</strong> {tags_html}</p>"
                
                html_output += "</div>"
                display(HTML(html_output))
            
            # Erstelle auch DataFrame für Export
            df = pd.DataFrame(all_annotations)
            print("\n📊 Daten als Tabelle:")
            display(df)
            
        except Exception as e:
            print(f"❌ Fehler: {e}")
            import traceback
            traceback.print_exc()

# Event-Handler
load_structure_button.on_click(load_structure)
group_dropdown.observe(update_collections, names='value')
load_items_button.on_click(load_items)
load_annotations_button.on_click(load_annotations)

Text(value='', description='User ID:', layout=Layout(width='50%'), placeholder='Deine Zotero User ID (z. B. 12…

Password(description='API-Key:', layout=Layout(width='50%'), placeholder='Dein API-Schlüssel')

Button(button_style='primary', description='Struktur laden', style=ButtonStyle())

Output()

In [6]:
def show_item_selector():
    global pdf_items
    
    # Dropdown mit PDFs
    options = [(f"{item['title']} ({item['filename']})", item['key']) for item in pdf_items]
    if not options:
        print("Keine PDFs gefunden.")
        return
    
    dropdown = widgets.Dropdown(
        options=options,
        description="PDF wählen:",
        layout=widgets.Layout(width='80%')
    )
    
    item_id_input = widgets.Text(
        placeholder='Oder direkt Item-Key eingeben',
        description='Item-Key:',
        layout=widgets.Layout(width='50%')
    )
    
    load_ann_button = widgets.Button(description="Highlights laden", button_style='primary')
    output_ann = widgets.Output()
    
    def on_load_ann(b):
        item_key = item_id_input.value.strip() or dropdown.value
        if not item_key:
            with output_ann:
                print("Bitte Item-Key eingeben oder auswählen!")
            return
        load_annotations(item_key, output_ann)
    
    load_ann_button.on_click(on_load_ann)
    
    display(dropdown, item_id_input, load_ann_button, output_ann)
    
    # Auto-load bei Auswahl
    def on_change(change):
        if change['name'] == 'value' and change['new']:
            item_id_input.value = change['new']
    dropdown.observe(on_change)

In [7]:
def load_annotations(item_key, output_widget):
    with output_widget:
        clear_output()
        print(f"Lade Annotationen für Item-Key: {item_key}...")
        
        url = f"{API_BASE}/users/{user_id_input.value.strip()}/items/{item_key}/children"
        params = {"itemType": "annotation"}
        
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            print(f"Fehler beim Laden der Annotationen: {response.status_code}")
            return
        
        annotations = response.json()
        highlights = [ann for ann in annotations if ann['data']['annotationType'] == 'highlight']
        
        if not highlights:
            print("Keine Highlights in diesem PDF gefunden.")
            return
        
        # Extrahiere Metadaten
        data = []
        for ann in highlights:
            d = ann['data']
            meta = ann.get('meta', {})
            
            row = {
                "Text": d.get('annotationText', '').strip(),
                "Seite": d.get('pageLabel', ''),
                "Farbe": d.get('annotationColor', ''),
                "Kommentar": d.get('annotationComment', '').strip(),
                "Tags": ", ".join([t['tag'] for t in d.get('tags', [])]),
                "Autor": meta.get('creatorSummary', ''),
                "Datum": meta.get('dateModified', '')[:10] if meta.get('dateModified') else '',
                "Position": d.get('annotationPosition', '')
            }
            data.append(row)
        
        df = pd.DataFrame(data)
        df.index = range(1, len(df) + 1)
        
        # Styling
        def color_row(row):
            color = row['Farbe']
            if color and len(color) == 7:
                return [f'background-color: {color}20' for _ in row]
            return [''] * len(row)
        
        styled = df.style.apply(color_row, axis=1).set_properties(**{
            'text-align': 'left',
            'white-space': 'pre-wrap'
        })
        
        display(HTML(f"<h3>{len(highlights)} Highlights gefunden</h3>"))
        display(styled)
        
        # Export-Button
        export_btn = widgets.Button(description="Als CSV exportieren", button_style='info')
        def export_csv(b):
            filename = f"zotero_highlights_{item_key}_{datetime.now().strftime('%Y%m%d')}.csv"
            df.to_csv(filename)
            print(f"Exportiert als: {filename}")
        export_btn.on_click(export_csv)
        display(export_btn)

## So verwendest du es:

1. **API-Zugang eintragen** (oben)
2. Klicke auf **"Bibliothek laden"**
3. Wähle ein PDF aus oder gib den **Item-Key** ein
4. Klicke auf **"Highlights laden"**
5. Optional: **CSV exportieren**

---
**Tipp:** Den Item-Key findest du in Zotero: Rechtsklick auf PDF → "Copy Item Key" oder in der URL beim Öffnen.